# 09 Hugging Face ViT Transfer Learning

## 1. Setup

This notebook fine-tunes a pretrained Hugging Face image-classification
transformer on CIFAR-100 fine labels. It is fully standalone: no local
package imports, no `%%writefile`, no shell git operations.

Run A is a frozen-backbone head baseline. Run B is a partial fine-tune.
Run C (optional) is LoRA via `peft`. LoRA is skipped gracefully if `peft`
fails to install or to wrap the chosen backbone.

[![Open 09 in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fgram-devAI/deepl-cifar100-image-analysis/blob/main/notebooks/09_hf_vit_transfer_learning.ipynb)

In [ ]:
%pip install -q --upgrade pip
%pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib

In [ ]:
RUN_LORA = False  # set to True to attempt LoRA Run C
PEFT_AVAILABLE = False

If you set `RUN_LORA = True`, run this dependency cell first:

In [ ]:
# Guarded equivalent of `%pip install -q peft`.
# Using run_line_magic keeps the cell valid Python when RUN_LORA is False.
if RUN_LORA:
    get_ipython().run_line_magic("pip", "install -q peft")

In [ ]:
if RUN_LORA:
    try:
        import peft  # noqa: F401
        PEFT_AVAILABLE = True
    except Exception as exc:
        print(f"PEFT import failed; LoRA Run C will be skipped: {exc}")
        PEFT_AVAILABLE = False

In [ ]:
import json
import os
import random
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

FAST_DEV_RUN = True       # small subsets, 1 epoch — used for smoke tests
RUN_TRAINING = True       # set False to only inspect data/model
MODEL_NAME = "google/vit-base-patch16-224-in21k"

OUTPUT_ROOT = Path("/content/hf_vit_transfer") if Path("/content").exists() else Path("results/hf_vit_transfer")
RUN_A_DIR = OUTPUT_ROOT / "run_a_frozen_head"
RUN_B_DIR = OUTPUT_ROOT / "run_b_partial_finetune"
RUN_C_DIR = OUTPUT_ROOT / "run_c_lora"

def ensure_dirs() -> None:
    for d in (RUN_A_DIR, RUN_B_DIR, RUN_C_DIR):
        d.mkdir(parents=True, exist_ok=True)

ensure_dirs()
print(f"output root: {OUTPUT_ROOT}")